# Getting started

`overviewpy` makes it easy to get an overview of a dataset by displaying relevant sample information.
This notebook walks through all four currently implemented functions:

| Function | What it does |
|---|---|
| `overview_tab` | Tabular overview — one row per id with collapsed time ranges |
| `overview_na` | Bar chart of missing-value counts or percentages (column- or row-wise) |
| `overview_summary` | Per-column summary: non-null count, unique count, sample values |
| `overview_plot` | Connected dot-plot showing id × time coverage |

All four are available as methods on the `Overview` class, which is the recommended interface.

## Import libraries

We import the `Overview` class from `overviewpy`, along with `pandas` for data manipulation and `numpy` to produce missing values in our example data.

In [ ]:
from overviewpy.overviewpy import Overview
import pandas as pd
import numpy as np

### Generate data

In the first step, we will generate some data that we will use in the next steps.

In [ ]:
# Generate full data

data = {
        'id': ['RWA', 'RWA', 'RWA', 'GAB', 'GAB', 'FRA',\
            'FRA', 'BEL', 'BEL', 'ARG'],
        'year': [2022, 2023, 2021, 2023, 2020, 2019, 2015,\
            2014, 2013, 2002]
    }

df = pd.DataFrame(data)

df.head()

We also generate a second data frame that contains missing values (`np.nan`) in both the `id` and `year` columns. This dataset is used to demonstrate `overview_na` and the NA-handling behaviour of `overview_tab`.

In [ ]:
# Generate data with missing values

data_na = {
        'id': ['RWA', 'RWA', 'RWA', np.nan, 'GAB', 'GAB',\
            'FRA', 'FRA', 'BEL', 'BEL', 'ARG', np.nan,  np.nan],
        'year': [2022, 2001, 2000, 2023, 2021, 2023, 2020,\
            2019,  np.nan, 2015, 2014, 2013, 2002]
    }

df_na = pd.DataFrame(data_na)

df_na.head()

### Get an overview of the time distribution in your data

Generate some general overview of the data set using the time and scope conditions with `overview_tab`. 
The resulting data frame collapses the time condition for each id by taking into account potential gaps in the time frame.

Consecutive years are compressed into a range (e.g. `2021-2023`); non-consecutive years appear as a comma-separated list (e.g. `2015, 2019`).

In [ ]:
overview = Overview(df=df, id='id', time='year')
df_overview = overview.overview_tab()

print(df_overview)

Missing values in `id` or `time` are automatically dropped and a `UserWarning` is raised for each affected variable. The code below captures those warnings explicitly so you can inspect them:

In [ ]:
import warnings

data_with_na = {
    'id': ['RWA', 'RWA', np.nan, 'GAB'],
    'year': [2022, np.nan, 2021, 2020],
}
df_with_na = pd.DataFrame(data_with_na)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    overview_with_na = Overview(df=df_with_na, id='id', time='year')
    df_overview_na = overview_with_na.overview_tab()

for w in caught:
    print(w.message)

print(df_overview_na)

### Get an overview of missing data in your data frame

`overview_na` visualises missing values in your data. It returns a horizontal bar chart
showing the share of NAs per variable (column-wise, the default) or per observation
(row-wise). You can also augment the original data frame with the computed NA counts
and percentages.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `yaxis` | `"Variables"` | y-axis label; auto-set to `"Observations"` when `row_wise=True` |
| `perc` | `True` | `True` = percentage of NAs, `False` = absolute counts |
| `row_wise` | `False` | `False` = per column, `True` = per row |
| `add` | `False` | when `True` (requires `row_wise=True`), returns the data frame extended with `na_count` and `percentage` instead of a plot |

In [ ]:
ov_na = Overview(df=df_na, id='id', time='year')

# Default: column-wise, percentage
ov_na.overview_na()

# Absolute counts instead of percentage
ov_na.overview_na(perc=False)

# Custom y-axis label
ov_na.overview_na(yaxis="My Variables")

# Row-wise: one bar per observation
ov_na.overview_na(row_wise=True)

# Row-wise and augment the data frame with na_count and percentage columns
ov_na.overview_na(row_wise=True, add=True)

### Get a column-level summary of your data frame

`overview_summary` returns a data frame with one row per column, showing:

* `non_null_count` — number of non-missing values
* `unique_count` — number of distinct non-missing values
* `sample_values` — up to 5 example values

It works on any data frame regardless of whether `id` and `time` are set.

In [ ]:
ov_na.overview_summary()

### Visualize observation coverage across id and time

`overview_plot` creates a connected dot-plot showing which id–time combinations are
present in the data:

* each id is a row on the y-axis
* time is on the x-axis
* **consecutive** years are joined by a line; **gaps** produce separate point clusters

This makes it easy to spot coverage gaps at a glance.

In [ ]:
# Basic coverage plot
overview.overview_plot()

#### Color-coding by a third variable

Pass any column name to `color` to color-code each point by that variable.
A legend is added automatically at the bottom of the plot.

In [ ]:
data_color = {
    'id':     ['RWA', 'RWA', 'RWA', 'GAB', 'GAB', 'FRA', 'FRA', 'BEL', 'BEL', 'ARG'],
    'year':   [2022,  2023,  2021,  2023,  2020,  2019,  2015,  2014,  2013,  2002],
    'regime': ['Dem', 'Dem', 'Aut', 'Aut', 'Aut', 'Dem', 'Dem', 'Dem', 'Dem', 'Aut'],
}

df_color = pd.DataFrame(data_color)
ov_color = Overview(df=df_color, id='id', time='year')

ov_color.overview_plot(color='regime')

#### Additional parameters

`overview_plot` exposes several parameters for fine-tuning the output:

| Parameter | Default | Description |
|---|---|---|
| `xaxis` | `"Time frame"` | x-axis label |
| `yaxis` | `"Sample"` | y-axis label |
| `asc` | `True` | Sort ids top-to-bottom in ascending order |
| `color` | `None` | Column name to color-code points by |
| `dot_size` | `2` | Size of the plotted squares |
| `show_plot` | `True` | Display the plot inline; set to `False` to return the axes object only |

In [ ]:
# Custom axis labels and descending id order
overview.overview_plot(
    xaxis='Year',
    yaxis='Country',
    asc=False,
)